# 🎂 סרטון מזל טוב לאבא
מרכיב סרטון מכל התמונות בתיקייה, עם שקופית פתיחה/סיום ומוזיקה.

**הרץ את התאים לפי הסדר.**

In [ ]:
# תא 1 — התקנת כלים
print('מתקין כלים...')
!apt-get install -y ffmpeg fonts-noto libheif-dev > /dev/null 2>&1
!pip install Pillow pillow-heif -q

import pillow_heif
pillow_heif.register_heif_opener()  # תמיכה ב-HEIC

from google.colab import drive
drive.mount('/content/drive')

import subprocess
font_check = subprocess.run("find /usr/share/fonts -name '*Hebrew*.ttf' | head -1",
                            shell=True, capture_output=True, text=True).stdout.strip()
print(f'פונט עברי: {font_check or "DejaVu (ברירת מחדל)"}')
print('✓ מוכן')

In [ ]:
# תא 2 — הגדרות נתיבים
import os, glob, shutil

# *** תיקיית התמונות החדשה ***
DRIVE_PHOTOS  = '/content/drive/MyDrive/עדכון תמונות לאבא'

# *** מוזיקה מהתיקייה הקודמת ***
DRIVE_MUSIC   = '/content/drive/MyDrive/Claude/יומולדת של אבא'

WORK_DIR      = '/content/mazel_tov'
PHOTOS_DIR    = f'{WORK_DIR}/photos_raw'
PHOTOS_FIXED  = f'{WORK_DIR}/photos_fixed'
VIDEOS_DIR    = f'{WORK_DIR}/videos'
MUSIC_DIR     = f'{WORK_DIR}/music'
OUTPUT_DIR    = f'{WORK_DIR}/output'
SEGMENTS_DIR  = f'{WORK_DIR}/segments'

for d in [PHOTOS_DIR, PHOTOS_FIXED, VIDEOS_DIR, MUSIC_DIR, OUTPUT_DIR, SEGMENTS_DIR]:
    os.makedirs(d, exist_ok=True)

ok1 = os.path.exists(DRIVE_PHOTOS)
ok2 = os.path.exists(DRIVE_MUSIC)
print(f'תיקיית תמונות: {"✓" if ok1 else "❌ לא נמצאה!"} {DRIVE_PHOTOS}')
print(f'תיקיית מוזיקה: {"✓" if ok2 else "❌ לא נמצאה!"} {DRIVE_MUSIC}')
if ok1:
    print(f'קבצים בתיקייה: {len(os.listdir(DRIVE_PHOTOS))}')

In [ ]:
# תא 3 — העתקת קבצים מ-Drive
IMAGE_EXTS = {'.jpg','.jpeg','.png','.heic','.JPG','.JPEG','.PNG','.HEIC'}
VIDEO_EXTS = {'.mp4','.MP4','.MOV','.mov','.avi','.AVI'}

photos_copied = videos_copied = 0

# תמונות + וידאו מתיקיית עדכון תמונות לאבא
for fname in sorted(os.listdir(DRIVE_PHOTOS)):
    src = os.path.join(DRIVE_PHOTOS, fname)
    if not os.path.isfile(src): continue
    ext = os.path.splitext(fname)[1]
    if ext in IMAGE_EXTS:
        dst = os.path.join(PHOTOS_DIR, fname)
        if not os.path.exists(dst): shutil.copy2(src, dst)
        photos_copied += 1
    elif ext in VIDEO_EXTS:
        dst = os.path.join(VIDEOS_DIR, fname)
        if not os.path.exists(dst): shutil.copy2(src, dst)
        videos_copied += 1

# מוזיקה
music_copied = 0
for fname in os.listdir(DRIVE_MUSIC):
    src = os.path.join(DRIVE_MUSIC, fname)
    if not os.path.isfile(src): continue
    ext = os.path.splitext(fname)[1]
    size = os.path.getsize(src)
    is_music = ext in VIDEO_EXTS and size < 5_000_000
    if is_music:
        dst = os.path.join(MUSIC_DIR, fname)
        if not os.path.exists(dst): shutil.copy2(src, dst)
        music_copied += 1

print(f'✓ תמונות: {photos_copied}')
print(f'✓ קטעי וידאו: {videos_copied}')
print(f'✓ שירים: {music_copied}  → {os.listdir(MUSIC_DIR)}')

In [ ]:
# תא 4 — תיקון סיבוב + חידוד תמונות מטושטשות
from PIL import Image, ImageOps, ImageFilter
import numpy as np

def blur_score(img):
    gray = np.array(img.convert('L'), dtype=float)
    return float(np.var(gray[1:]-gray[:-1]) + np.var(gray[:,1:]-gray[:,:-1]))

photos_raw = sorted(glob.glob(f'{PHOTOS_DIR}/*'))
print(f'סה"כ קבצים שהועתקו: {len(photos_raw)} (צפוי: 117)')

fixed_paths = []
sharpened = errors = 0

for i, src in enumerate(photos_raw):
    # מדלגים רק על קבצים שבורים לגמרי (< 5KB)
    if os.path.getsize(src) < 5 * 1024:
        print(f'  ⚠️  דולג (קובץ קטן מ-5KB): {os.path.basename(src)}')
        continue

    base = os.path.splitext(os.path.basename(src))[0] + '.jpg'
    dst = os.path.join(PHOTOS_FIXED, base)

    if not os.path.exists(dst):
        try:
            img = Image.open(src)
            img = ImageOps.exif_transpose(img)  # תיקון סיבוב EXIF
            img = img.convert('RGB')

            # חידוד לתמונות מטושטשות
            if blur_score(img) < 500:
                img = img.filter(ImageFilter.UnsharpMask(radius=2, percent=180, threshold=3))
                sharpened += 1

            img.save(dst, 'JPEG', quality=93)
        except Exception as e:
            print(f'  ⚠️  {os.path.basename(src)}: {e}')
            try:
                ext = os.path.splitext(src)[1]
                dst = os.path.join(PHOTOS_FIXED, os.path.basename(src))
                shutil.copy2(src, dst)
            except:
                errors += 1
                continue

    fixed_paths.append(dst)
    if (i+1) % 20 == 0:
        print(f'  עובד... {i+1}/{len(photos_raw)}')

print(f'\n✓ {len(fixed_paths)}/117 תמונות מוכנות')
print(f'   חודדו: {sharpened} | שגיאות: {errors}')
if len(fixed_paths) < 117:
    print(f'  ⚠️  חסרות {117 - len(fixed_paths)} תמונות — בדוק שגיאות למעלה')

In [ ]:
# תא 5 — הגדרת מוזיקה (שנה סדר אם צריך)
music_files = sorted(glob.glob(f'{MUSIC_DIR}/*'))
print('שירים שנמצאו:')
for i, f in enumerate(music_files):
    print(f'  [{i}] {os.path.basename(f)}')

if not music_files:
    raise Exception('❌ לא נמצאה מוזיקה!')

MUSIC1 = music_files[0]
MUSIC2 = music_files[1] if len(music_files) > 1 else music_files[0]
print(f'\nשיר 1: {os.path.basename(MUSIC1)}')
print(f'שיר 2: {os.path.basename(MUSIC2)}')

In [ ]:
# תא 6 — פונקציות בנייה
import subprocess

W, H = 1920, 1080
FPS  = 25
PHOTO_DURATION = 5

TITLE_TEXT = 'אבא היקר\nהמון מזל טוב'
END_TEXT   = 'המון מזל טוב\nאוהבים אותך\nמשפחת פיינגרש'

def run(cmd, desc=''):
    print(f'  → {desc}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ❌ {r.stderr[-300:]}')
        return False
    return True

def find_hebrew_font():
    for f in [
        '/usr/share/fonts/truetype/noto/NotoSansHebrew-Regular.ttf',
        '/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    ]:
        if os.path.exists(f): return f
    r = subprocess.run("find /usr/share/fonts -name '*.ttf' | head -1",
                       shell=True, capture_output=True, text=True)
    return r.stdout.strip()

def get_duration(path):
    r = subprocess.run(
        f'ffprobe -v quiet -show_entries format=duration -of csv=p=0 "{path}"',
        shell=True, capture_output=True, text=True)
    try: return float(r.stdout.strip())
    except: return PHOTO_DURATION

def make_title_slide(text, filename, duration=8, bg='1a0a00'):
    out = f'{SEGMENTS_DIR}/{filename}'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    font  = find_hebrew_font()
    lines = text.split('\n')
    line_h  = 110
    start_y = (H - len(lines)*line_h) // 2
    parts   = []
    for i, line in enumerate(lines):
        esc   = line.replace("'", "\\'").replace(':', '\\:')
        size  = 96 if i == 0 else 78
        color = 'white' if i == 0 else '#FFD700'
        y     = start_y + i*line_h
        parts.append(
            f"drawtext=fontfile='{font}':text='{esc}':fontcolor={color}:"
            f"fontsize={size}:x=(w-text_w)/2:y={y}:"
            f"shadowcolor=black:shadowx=3:shadowy=3"
        )
    vf = ','.join(parts) + f',fade=t=in:st=0:d=1.5,fade=t=out:st={duration-1.5}:d=1.5'
    cmd = (f'ffmpeg -y -f lavfi -i color=c={bg}:{W}x{H}:rate={FPS}:duration={duration} '
           f'-vf "{vf}" -c:v libx264 -preset fast -crf 18 -pix_fmt yuv420p "{out}"')
    run(cmd, f'שקופית: {filename}')
    return out

def make_photo_segment(photo_path, idx, duration=PHOTO_DURATION):
    out = f'{SEGMENTS_DIR}/photo_{idx:03d}.mp4'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    BIG_W, BIG_H  = 2112, 1188
    total_frames  = duration * FPS
    scale = (
        f'scale={BIG_W}:{BIG_H}:force_original_aspect_ratio=decrease,'
        f'pad={BIG_W}:{BIG_H}:(ow-iw)/2:(oh-ih)/2:black,'
        f'scale=trunc(iw/2)*2:trunc(ih/2)*2'
    )
    if   idx % 3 == 0: z = "'min(zoom+0.0006,1.05)'"
    elif idx % 3 == 1: z = "'if(lte(zoom,1.0),1.04,max(1.0,zoom-0.0006))'"
    else:              z = "'1.02'"
    zoom = (f"zoompan=z={z}:d={total_frames}:"
            f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':s={W}x{H}:fps={FPS}")
    fade = f'fade=t=in:st=0:d=0.5,fade=t=out:st={duration-0.5}:d=0.5'
    cmd  = (
        f'ffmpeg -y -loop 1 -i "{photo_path}" '
        f'-vf "{scale},{zoom},{fade}" '
        f'-t {duration} -r {FPS} -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p "{out}"'
    )
    ok = run(cmd, f'תמונה {idx+1}: {os.path.basename(photo_path)}')
    if not ok and os.path.exists(out) and os.path.getsize(out) == 0:
        os.remove(out)
    return out

def make_video_segment(video_path, idx, max_dur=25):
    out = f'{SEGMENTS_DIR}/video_{idx:03d}.mp4'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    dur   = get_duration(video_path)
    trim  = min(dur, max_dur)
    start = (dur - max_dur) / 4 if dur > max_dur else 0
    fo    = max(0, trim - 0.5)
    vf    = (
        f'scale={W}:{H}:force_original_aspect_ratio=decrease,'
        f'pad={W}:{H}:(ow-iw)/2:(oh-ih)/2:black,'
        f'fade=t=in:st=0:d=0.5,fade=t=out:st={fo}:d=0.5'
    )
    cmd = (
        f'ffmpeg -y -ss {start:.2f} -i "{video_path}" -t {trim:.2f} '
        f'-vf "{vf}" -r {FPS} -c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p -an "{out}"'
    )
    run(cmd, f'וידאו {idx+1}: {os.path.basename(video_path)}')
    return out

def mix_music(total_dur):
    out = f'{WORK_DIR}/mixed_audio.aac'
    if os.path.exists(out) and os.path.getsize(out) > 0: return out
    d1 = get_duration(MUSIC1)
    d2 = get_duration(MUSIC2)
    p1 = min(total_dur * 0.55, d1)
    p2 = min(total_dur - p1, d2)
    fade = 3.0
    cmd = (
        f'ffmpeg -y -i "{MUSIC1}" -i "{MUSIC2}" -filter_complex '
        f'"[0:a]atrim=0:{p1},asetpts=PTS-STARTPTS,'
        f'afade=t=in:st=0:d={fade},afade=t=out:st={p1-fade}:d={fade}[a1];'
        f'[1:a]atrim=0:{p2},asetpts=PTS-STARTPTS,'
        f'afade=t=in:st=0:d={fade},afade=t=out:st={p2-fade}:d={fade}[a2];'
        f'[a1][a2]concat=n=2:v=0:a=1[aout]" '
        f'-map [aout] -c:a aac -b:a 192k "{out}"'
    )
    run(cmd, f'מיקס מוזיקה ({p1:.0f}ש + {p2:.0f}ש)')
    return out

print(f'✓ פונקציות מוגדרות | פונט: {find_hebrew_font()}')

In [ ]:
# תא 7 — בנייה (~20-45 דקות)
photos = fixed_paths
videos = sorted(
    glob.glob(f'{VIDEOS_DIR}/*.mp4') + glob.glob(f'{VIDEOS_DIR}/*.MP4') +
    glob.glob(f'{VIDEOS_DIR}/*.MOV') + glob.glob(f'{VIDEOS_DIR}/*.mov')
)
print(f'תמונות: {len(photos)} | קטעי וידאו: {len(videos)}')

segments = []

# שקף פתיחה
print('\n--- שקף פתיחה ---')
seg = make_title_slide(TITLE_TEXT, 'title.mp4', duration=8, bg='0a0a1a')
if os.path.exists(seg) and os.path.getsize(seg) > 0:
    segments.append(seg)

# תמונות + קטעי וידאו (כל 6 תמונות — קטע וידאו)
print('\n--- תמונות וסרטונים ---')
vi = 0
for i, photo in enumerate(photos):
    seg = make_photo_segment(photo, i)
    if os.path.exists(seg) and os.path.getsize(seg) > 0:
        segments.append(seg)
    if (i+1) % 6 == 0 and vi < len(videos):
        seg = make_video_segment(videos[vi], vi)
        if os.path.exists(seg) and os.path.getsize(seg) > 0:
            segments.append(seg)
        vi += 1

# קטעי וידאו שנותרו
while vi < len(videos):
    seg = make_video_segment(videos[vi], vi)
    if os.path.exists(seg) and os.path.getsize(seg) > 0:
        segments.append(seg)
    vi += 1

# שקף סיום
print('\n--- שקף סיום ---')
seg = make_title_slide(END_TEXT, 'end.mp4', duration=10, bg='1a0a00')
if os.path.exists(seg) and os.path.getsize(seg) > 0:
    segments.append(seg)

total_dur = sum(get_duration(s) for s in segments)
print(f'\n✓ {len(segments)} סגמנטים | {total_dur:.0f}ש ({total_dur/60:.1f} דקות)')

In [ ]:
# תא 8 — שרשור + מוזיקה
list_file = f'{WORK_DIR}/concat_list.txt'
with open(list_file, 'w') as f:
    for s in segments:
        f.write(f"file '{s}'\n")

silent = f'{WORK_DIR}/silent_video.mp4'
run(
    f'ffmpeg -y -f concat -safe 0 -i "{list_file}" '
    f'-c:v libx264 -preset fast -crf 20 -pix_fmt yuv420p "{silent}"',
    'שרשור סגמנטים'
)

music = mix_music(total_dur)

FINAL = f'{OUTPUT_DIR}/mazel_tov_final.mp4'
run(
    f'ffmpeg -y -i "{silent}" -i "{music}" '
    f'-c:v copy -c:a aac -b:a 192k -shortest "{FINAL}"',
    'הוספת מוזיקה'
)

size = os.path.getsize(FINAL) / 1024 / 1024
dur  = get_duration(FINAL)
print(f'\n✓ הסרטון מוכן! {size:.0f}MB | {dur/60:.1f} דקות')

In [ ]:
# תא 9 — שמירה חזרה ל-Drive
import shutil

DRIVE_OUT = f'{DRIVE_PHOTOS}/mazel_tov_final.mp4'
print('שומר ל-Drive...')
shutil.copy2(FINAL, DRIVE_OUT)
print(f'✓ נשמר: {DRIVE_OUT}')
print(f'   גודל: {os.path.getsize(DRIVE_OUT)/1024/1024:.0f}MB')